In [ ]:
# %% [markdown]
# # 01 - Data Preprocessing
# ## Customer Segmentation Project

# %% [markdown]
# ### Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

# %% [markdown]
# ### Load Dataset
# Load the Online Retail II dataset
data_path = Path("../data/raw/online_retail_II.csv")
df = pd.read_csv(data_path, encoding='latin1')

print(f"Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Time range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")

# %% [markdown]
# ### Initial Data Exploration
# Show first few rows
display(df.head())

# Show data types and missing values
print("\nData Information:")
print(df.info())

# Show basic statistics
print("\nBasic Statistics:")
display(df.describe())

# %% [markdown]
# ### Data Quality Assessment
# Check for missing values
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage': missing_percentage
})

print("Missing Values Summary:")
display(missing_df[missing_df['Missing Values'] > 0])

# %% [markdown]
# ### Data Cleaning Pipeline
from src.data_preprocessing import DataPreprocessor

# Initialize preprocessor
preprocessor = DataPreprocessor()

# Clean the data
df_clean = preprocessor.clean_data(df)

# %% [markdown]
# ### Save Cleaned Data
# Save to processed folder
output_path = Path("../data/processed/cleaned_data.csv")
df_clean.to_csv(output_path, index=False)

print(f"\nCleaned data saved to: {output_path}")
print(f"Original shape: {df.shape}")
print(f"Cleaned shape: {df_clean.shape}")

# %% [markdown]
# ### Data Quality Report
# Generate cleaning report
report = preprocessor.get_summary()

print("\n" + "="*70)
print("DATA CLEANING REPORT")
print("="*70)
print(f"Original records: {report['cleaning_stats']['original_records']:,}")
print(f"Final records: {report['cleaning_stats']['final_records']:,}")
print(f"Records removed: {report['cleaning_stats']['removed_records']:,}")
print(f"Removal percentage: {report['cleaning_stats']['removed_percentage']:.1f}%")
print("="*70)

# %% [markdown]
# ### Visualize Data Distribution
# Create visualizations for key columns
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Quantity distribution
axes[0, 0].hist(df_clean['Quantity'].values, bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Quantity')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Quantity Distribution')
axes[0, 0].grid(True, alpha=0.3)

# Unit price distribution
axes[0, 1].hist(df_clean['UnitPrice'].values, bins=50, edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Unit Price (£)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Unit Price Distribution')
axes[0, 1].grid(True, alpha=0.3)

# Total value distribution
if 'TotalValue' in df_clean.columns:
    axes[1, 0].hist(df_clean['TotalValue'].values, bins=50, edgecolor='black', alpha=0.7)
    axes[1, 0].set_xlabel('Total Value (£)')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Transaction Value Distribution')
    axes[1, 0].grid(True, alpha=0.3)

# Country distribution
country_counts = df_clean['Country'].value_counts().head(10)
axes[1, 1].barh(country_counts.index, country_counts.values)
axes[1, 1].set_xlabel('Number of Transactions')
axes[1, 1].set_title('Top 10 Countries by Transaction Count')
axes[1, 1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('../results/cluster_plots/data_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# %% [markdown]
# ### Summary
print("\n✅ Data preprocessing completed successfully!")
print(f"📁 Cleaned data saved to: data/processed/cleaned_data.csv")
print(f"📊 Visualizations saved to: results/cluster_plots/")